In [0]:
from pyspark.sql import Row
from datetime import date, timedelta

# Helper to generate consecutive dates
def d(days_offset):
    return date(2025, 1, 1) + timedelta(days=days_offset)

data = [
    # User 1: Has a 6-day streak of 'post'
    Row(user_id=1, action_date=d(0), action='post'),
    Row(user_id=1, action_date=d(1), action='post'),
    Row(user_id=1, action_date=d(2), action='post'),
    Row(user_id=1, action_date=d(3), action='post'),
    Row(user_id=1, action_date=d(4), action='post'),
    Row(user_id=1, action_date=d(5), action='post'),
    
    # User 2: Has two streaks (4 days and 5 days) - only the 5 should show
    Row(user_id=2, action_date=d(0), action='like'),
    Row(user_id=2, action_date=d(1), action='like'),
    Row(user_id=2, action_date=d(2), action='like'),
    Row(user_id=2, action_date=d(3), action='like'),
    # Gap here
    Row(user_id=2, action_date=d(10), action='like'),
    Row(user_id=2, action_date=d(11), action='like'),
    Row(user_id=2, action_date=d(12), action='like'),
    Row(user_id=2, action_date=d(13), action='like'),
    Row(user_id=2, action_date=d(14), action='like'),
    
    # User 3: Only 3 days (Should be excluded)
    Row(user_id=3, action_date=d(0), action='post'),
    Row(user_id=3, action_date=d(1), action='post'),
    Row(user_id=3, action_date=d(2), action='post'),
]

df = spark.createDataFrame(data)
df.createOrReplaceTempView("activity")

# Show the data
df.show()

In [0]:
spark.sql(
    """
with grouped_date as (
SELECT *,
    DATE_SUB(action_date, INTERVAL ROW_NUMBER() OVER(PARTITION BY user_id, action ORDER BY action_date) DAY) AS group_date
FROM activity )

,calculated as (
select user_id, action,
        count(*) as streak_length,
        min(action_date) as start_date, 
        max(action_date) as end_date
    from grouped_date
group by user_id, action, group_date
having streak_length >= 5)

,ranked as (
    select *, RANK() over(partition by user_id order by streak_length DESC) as rnk from calculated
)

select user_id, action, streak_length, start_date, end_date  from ranked where rnk=1 order by streak_length DESC, user_id ASC
""").show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

grouped_df = df.withColumn("group_date", F.date_sub(F.col("action_date"), F.row_number().over(Window.partitionBy("user_id", "action").orderBy(F.col("action_date")))))

calculated_df = grouped_df.groupBy("user_id", "action", "group_date"). \
        agg(F.count("*").alias("streak_length"),
            F.min("action_date").alias("start_date"),
            F.max("action_date").alias("end_date")) \
                .filter(F.col("streak_length") >= 5)
ranked = calculated_df.withColumn('rnk', F.rank().over(Window.partitionBy("user_id").orderBy(F.col("streak_length").desc()))).filter(F.col("rnk") == 1).orderBy(F.col("streak_length").desc(), F.col("user_id").asc())
display(ranked)



# Group by user_id, action, and group_date)